# Split por Rótulos com Chunking

Segmenta os logs extraídos em classes (`square`, `front_to_back`, `side_to_side`, `shoot_to_goal`) e **quebra intervalos longos em sub-eventos de duração máxima** (`CHUNK_MAX_SEC`).

Estratégia de chunking: cada anotação de duração `D` vira `N = ⌈D / CHUNK_MAX_SEC⌉` sub-intervalos de tamanho **igual** `D/N`. Sem sobreposição, sem descarte de resto.

**event_id**: `<label>_<annotation>_<chunk>` (ex.: `square_01_03` = label `square`, 1ª anotação, 3º chunk).

**Saídas:**
- `splits/<label>/<event_id>/<stream>.{csv,parquet}` — um diretório por sub-evento
- `splits/all_<stream>_labeled.parquet` — tabela concatenada por stream com `label`, `event_id`, `source_idx`, `chunk_idx`
- `splits/labels_manifest.csv` — metadados (duração, linhas por stream, lineage)

## 1. Setup

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# ------------------ CONFIG ------------------
EXTRACTED_DIR = Path('extracted/2026-05-18_19-2-15')   # troque pelo log
SPLITS_DIR    = EXTRACTED_DIR / 'splits'

CHUNK_MAX_SEC = 10.0          # 10 s -> bom equilíbrio; troque pra 5.0 se quiser mais amostras

STREAMS = ['ball', 'raw_ball', 'processed_robots', 'raw_robots',
           'commands', 'telemetry', 'referee']

SAVE_PER_EVENT   = True
SAVE_CONCAT      = True
REBASE_TIMESTAMP = True       # acrescenta coluna timestamp_event = timestamp - t_start
# --------------------------------------------

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
print('Lendo de:', EXTRACTED_DIR.resolve())
print(f'Chunk máximo: {CHUNK_MAX_SEC}s')

Lendo de: C:\Users\victo\Documents\digitaltwin\scripts\local\extracted\2026-05-18_19-2-15
Chunk máximo: 10.0s


## 2. Carregar os streams

In [2]:
def _load_stream(name: str) -> pd.DataFrame:
    pq = EXTRACTED_DIR / f'{name}.parquet'
    cs = EXTRACTED_DIR / f'{name}.csv'
    if pq.exists(): return pd.read_parquet(pq)
    if cs.exists(): return pd.read_csv(cs)
    return pd.DataFrame()

dfs = {name: _load_stream(name) for name in STREAMS}
for name, df in dfs.items():
    print(f'{name:>20s}: {len(df):>8d} linhas')

                ball:    27413 linhas
            raw_ball:    14150 linhas
    processed_robots:    27408 linhas
          raw_robots:    16131 linhas
            commands:    48012 linhas
           telemetry:    57415 linhas
             referee:    27407 linhas


## 3. Anotações (intervalos brutos)

Tempos em segundos a partir de `t=0` do log.

In [3]:
intervals_raw = pd.DataFrame([
    # ---- square ----
    {'label': 'square',        't_start':   30.225, 't_end':   83.991},
    {'label': 'square',        't_start': 1310.665, 't_end': 1356.748},
    # ---- front_to_back ----
    {'label': 'front_to_back', 't_start':  497.298, 't_end':  530.279},
    # ---- side_to_side ----
    {'label': 'side_to_side',  't_start':  341.668, 't_end':  420.277},
    {'label': 'side_to_side',  't_start':  454.223, 't_end':  496.798},
    # ---- shoot_to_goal ----
    {'label': 'shoot_to_goal', 't_start':  611.654, 't_end':  616.548},
    {'label': 'shoot_to_goal', 't_start':  617.085, 't_end':  621.449},
    {'label': 'shoot_to_goal', 't_start':  622.657, 't_end':  630.929},
    {'label': 'shoot_to_goal', 't_start':  631.534, 't_end':  640.345},
    {'label': 'shoot_to_goal', 't_start':  659.762, 't_end':  671.415},
    {'label': 'shoot_to_goal', 't_start':  709.028, 't_end':  713.959},
    {'label': 'shoot_to_goal', 't_start':  713.412, 't_end':  720.057},  # sobrepõe com anterior
    {'label': 'shoot_to_goal', 't_start':  721.290, 't_end':  726.117},
    {'label': 'shoot_to_goal', 't_start':  728.569, 't_end':  736.909},
    {'label': 'shoot_to_goal', 't_start':  737.314, 't_end':  742.809},
    {'label': 'shoot_to_goal', 't_start':  743.416, 't_end':  760.715},
    {'label': 'shoot_to_goal', 't_start':  780.613, 't_end':  790.155},
    {'label': 'shoot_to_goal', 't_start':  806.835, 't_end':  815.077},
    {'label': 'shoot_to_goal', 't_start':  815.413, 't_end':  829.618},
    {'label': 'shoot_to_goal', 't_start':  830.824, 't_end':  839.775},
])

intervals_raw['duration'] = intervals_raw['t_end'] - intervals_raw['t_start']
intervals_raw

,label,t_start,t_end,duration
0,square,30.225,83.991,53.766
1,square,1310.665,1356.748,46.083
2,front_to_back,497.298,530.279,32.981
3,side_to_side,341.668,420.277,78.609
4,side_to_side,454.223,496.798,42.575
5,shoot_to_goal,611.654,616.548,4.894
6,shoot_to_goal,617.085,621.449,4.364
7,shoot_to_goal,622.657,630.929,8.272
8,shoot_to_goal,631.534,640.345,8.811
9,shoot_to_goal,659.762,671.415,11.653


## 4. Chunking — quebrar intervalos longos em sub-eventos iguais

Para cada anotação de duração `D`:
- `N = max(1, ⌈D / CHUNK_MAX_SEC⌉)`
- duração de cada chunk = `D / N` (uniforme, contígua, sem sobreposição)

Exemplo: anotação de 53,77 s com `CHUNK_MAX_SEC=10` → 6 chunks de 8,96 s.

In [4]:
def chunk_intervals(intervals: pd.DataFrame, max_sec: float) -> pd.DataFrame:
    rows = []
    # source_idx é por label (não global) para event_id fazer sentido
    for label, grp in intervals.groupby('label', sort=False):
        for src_idx, (_, ev) in enumerate(grp.iterrows(), start=1):
            t0, t1 = float(ev['t_start']), float(ev['t_end'])
            dur    = t1 - t0
            n      = max(1, int(np.ceil(dur / max_sec)))
            cdur   = dur / n
            for ck in range(1, n + 1):
                rows.append({
                    'label':       label,
                    'source_idx':  src_idx,
                    'chunk_idx':   ck,
                    'n_chunks':    n,
                    't_start':     t0 + (ck - 1) * cdur,
                    't_end':       t0 + ck * cdur,
                    'duration':    cdur,
                })
    out = pd.DataFrame(rows)
    _safe = lambda s: re.sub(r'[^A-Za-z0-9_\-]+', '_', str(s))
    out['event_id'] = [
        f'{_safe(l)}_{s:02d}_{c:02d}'
        for l, s, c in zip(out['label'], out['source_idx'], out['chunk_idx'])
    ]
    return out

intervals = chunk_intervals(intervals_raw, CHUNK_MAX_SEC)
print(f'Anotações originais: {len(intervals_raw)}  ->  sub-eventos após chunking: {len(intervals)}')
intervals.head(15)

Anotações originais: 20  ->  sub-eventos após chunking: 46


,label,source_idx,chunk_idx,n_chunks,t_start,t_end,duration,event_id
0,square,1,1,6,30.22500,39.18600,8.96100,square_01_01
1,square,1,2,6,39.18600,48.14700,8.96100,square_01_02
2,square,1,3,6,48.14700,57.10800,8.96100,square_01_03
3,square,1,4,6,57.10800,66.06900,8.96100,square_01_04
4,square,1,5,6,66.06900,75.03000,8.96100,square_01_05
5,square,1,6,6,75.03000,83.99100,8.96100,square_01_06
6,square,2,1,5,1310.66500,1319.88160,9.21660,square_02_01
7,square,2,2,5,1319.88160,1329.09820,9.21660,square_02_02
8,square,2,3,5,1329.09820,1338.31480,9.21660,square_02_03
9,square,2,4,5,1338.31480,1347.53140,9.21660,square_02_04


In [5]:
# Resumo por label
summary = (intervals.groupby('label')
                    .agg(n_events=('event_id','count'),
                         n_sources=('source_idx','nunique'),
                         total_sec=('duration','sum'),
                         mean_sec=('duration','mean'),
                         min_sec=('duration','min'),
                         max_sec=('duration','max'))
                    .round(2)
                    .sort_values('n_events', ascending=False))
summary

,n_events,n_sources,total_sec,mean_sec,min_sec,max_sec
label,,,,,,
shoot_to_goal,18,15,126.47,7.03,4.36,9.54
side_to_side,13,2,121.18,9.32,8.51,9.83
square,11,2,99.85,9.08,8.96,9.22
front_to_back,4,1,32.98,8.25,8.25,8.25


## 5. Validação

In [6]:
def validate_intervals(intervals: pd.DataFrame, dfs: dict) -> None:
    assert (intervals['t_end'] > intervals['t_start']).all(), 't_end <= t_start em alguma linha'

    t_min = min(d['timestamp'].iloc[0]  for d in dfs.values() if not d.empty)
    t_max = max(d['timestamp'].iloc[-1] for d in dfs.values() if not d.empty)
    out_of = intervals[(intervals['t_start'] < t_min) | (intervals['t_end'] > t_max)]
    if not out_of.empty:
        print(f'⚠️  {len(out_of)} chunks fora do log (log: [{t_min:.2f}, {t_max:.2f}]s):')
        print(out_of[['event_id','t_start','t_end']])
    else:
        print(f'✓ todos os {len(intervals)} chunks dentro do log [{t_min:.2f}, {t_max:.2f}]s')

    # sobreposições entre anotações originais (chunks internos NÃO se sobrepõem por construção)
    src = (intervals.groupby(['label','source_idx'])
                    .agg(t_start=('t_start','min'), t_end=('t_end','max'))
                    .reset_index()
                    .sort_values('t_start').reset_index(drop=True))
    overlaps = []
    for i in range(len(src) - 1):
        if src.loc[i, 't_end'] > src.loc[i + 1, 't_start']:
            overlaps.append((f"{src.loc[i,'label']}#{src.loc[i,'source_idx']}",
                             f"{src.loc[i+1,'label']}#{src.loc[i+1,'source_idx']}"))
    if overlaps:
        print(f'\nℹ️  sobreposições entre anotações (ok, só registrando):')
        for a, b in overlaps:
            print(f'  {a}  ↔  {b}')

validate_intervals(intervals, dfs)

✓ todos os 46 chunks dentro do log [0.00, 1356.76]s

ℹ️  sobreposições entre anotações (ok, só registrando):
  shoot_to_goal#6  ↔  shoot_to_goal#7


## 6. Split

In [7]:
_safe = lambda s: re.sub(r'[^A-Za-z0-9_\-]+', '_', str(s))

def split_dataset(dfs: dict, intervals: pd.DataFrame, out_root: Path,
                  save_per_event=True, save_concat=True, rebase=True) -> pd.DataFrame:
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    manifest = []
    concat = {name: [] for name in dfs}

    for _, ev in intervals.iterrows():
        t0, t1   = float(ev['t_start']), float(ev['t_end'])
        label    = ev['label']
        event_id = ev['event_id']

        rec = {'event_id': event_id, 'label': label,
               'source_idx': int(ev['source_idx']),
               'chunk_idx':  int(ev['chunk_idx']),
               'n_chunks':   int(ev['n_chunks']),
               't_start': t0, 't_end': t1, 'duration': t1 - t0}

        event_dir = out_root / _safe(label) / event_id
        if save_per_event:
            event_dir.mkdir(parents=True, exist_ok=True)

        for name, df in dfs.items():
            if df.empty or 'timestamp' not in df.columns:
                rec[f'{name}_rows'] = 0
                continue

            # [t0, t1) -> evita dupla contagem na fronteira entre chunks contíguos
            mask = (df['timestamp'] >= t0) & (df['timestamp'] < t1)
            sub  = df.loc[mask].copy()
            if rebase and not sub.empty:
                sub['timestamp_event'] = sub['timestamp'] - t0

            rec[f'{name}_rows'] = int(len(sub))

            if save_per_event and not sub.empty:
                sub.to_csv(event_dir / f'{name}.csv', index=False)
                sub.to_parquet(event_dir / f'{name}.parquet', index=False)

            if save_concat and not sub.empty:
                sub['label']      = label
                sub['event_id']   = event_id
                sub['source_idx'] = int(ev['source_idx'])
                sub['chunk_idx']  = int(ev['chunk_idx'])
                concat[name].append(sub)

        manifest.append(rec)

    manifest_df = pd.DataFrame(manifest)
    manifest_df.to_csv(out_root / 'labels_manifest.csv', index=False)

    if save_concat:
        for name, chunks in concat.items():
            if not chunks:
                continue
            big = pd.concat(chunks, ignore_index=True)
            big.to_parquet(out_root / f'all_{name}_labeled.parquet', index=False)

    return manifest_df

manifest = split_dataset(dfs, intervals, SPLITS_DIR,
                         save_per_event=SAVE_PER_EVENT,
                         save_concat=SAVE_CONCAT,
                         rebase=REBASE_TIMESTAMP)
print(f'\n✅ {len(manifest)} sub-eventos salvos em {SPLITS_DIR.resolve()}')
manifest.head(10)


✅ 46 sub-eventos salvos em C:\Users\victo\Documents\digitaltwin\scripts\local\extracted\2026-05-18_19-2-15\splits


,event_id,label,source_idx,chunk_idx,n_chunks,t_start,t_end,duration,ball_rows,raw_ball_rows,processed_robots_rows,raw_robots_rows,commands_rows,telemetry_rows,referee_rows
0,square_01_01,square,1,1,6,30.2250,39.1860,8.9610,508,237,508,273,1051,1118,508
1,square_01_02,square,1,2,6,39.1860,48.1470,8.9610,514,236,514,268,1055,1127,514
2,square_01_03,square,1,3,6,48.1470,57.1080,8.9610,509,236,509,287,1060,1129,509
3,square_01_04,square,1,4,6,57.1080,66.0690,8.9610,515,242,515,289,1061,1128,515
4,square_01_05,square,1,5,6,66.0690,75.0300,8.9610,517,244,517,276,1062,1129,517
5,square_01_06,square,1,6,6,75.0300,83.9910,8.9610,516,263,516,294,1061,1129,516
6,square_02_01,square,2,1,5,1310.6650,1319.8816,9.2166,475,16,475,241,1077,1155,475
7,square_02_02,square,2,2,5,1319.8816,1329.0982,9.2166,478,110,478,243,1084,1155,478
8,square_02_03,square,2,3,5,1329.0982,1338.3148,9.2166,483,69,483,261,1086,1155,482
9,square_02_04,square,2,4,5,1338.3148,1347.5314,9.2166,483,113,483,255,1087,1162,483


## 7. Sanidade — quantas linhas cada stream recebeu por label?

In [8]:
row_cols = [c for c in manifest.columns if c.endswith('_rows')]
by_label = manifest.groupby('label')[row_cols].sum()
by_label['n_events'] = manifest.groupby('label').size()
by_label = by_label[['n_events'] + row_cols]
by_label

,n_events,ball_rows,raw_ball_rows,processed_robots_rows,raw_robots_rows,commands_rows,telemetry_rows,referee_rows
label,,,,,,,,
front_to_back,4,1822,960,1822,941,3877,4150,1821
shoot_to_goal,18,6611,4765,6611,4438,7519,15778,6607
side_to_side,13,6785,3226,6785,3549,14281,15247,6782
square,11,5478,1844,5478,2943,11760,12542,5477


In [9]:
# alerta para chunks com pouquíssimas amostras (provável buraco no log naquele intervalo)
ball_rows = manifest['ball_rows']
thin = manifest[ball_rows < 30][['event_id','t_start','t_end','duration','ball_rows','processed_robots_rows']]
if not thin.empty:
    print(f'⚠️  {len(thin)} chunks com < 30 amostras de bola — verificar:')
    print(thin)
else:
    print('✓ todos os chunks com ≥ 30 amostras de bola')

✓ todos os chunks com ≥ 30 amostras de bola


## 8. Receitas de uso

**Carregar todos os chutes de uma vez:**
```python
big = pd.read_parquet('extracted/<log>/splits/all_processed_robots_labeled.parquet')
chutes = big[big['label'] == 'shoot_to_goal']

for ev_id, ev_df in chutes.groupby('event_id'):
    # ev_df.timestamp_event começa em 0 -> séries comparáveis
    ...
```

**Reconstruir uma anotação inteira (juntar chunks):**
```python
big_ball = pd.read_parquet('.../all_ball_labeled.parquet')
ann = big_ball[(big_ball['label']=='square') & (big_ball['source_idx']==1)]
ann = ann.sort_values('timestamp')
```

**Carregar um chunk específico:**
```python
ball = pd.read_parquet('extracted/<log>/splits/square/square_01_03/ball.parquet')
```